# Generate synthetic designs

**LD-BT 3 synthetic-design step.** Build the Round 3 candidate motifs by constrained mutation: amino acids observed at or above wild-type activity define a per-position residue set, and new motifs are sampled by applying a bounded number of those allowed substitutions to templates drawn from the earlier rounds. The sampled motifs are then scored by the two-stage model (`src/ml/two_stage.py`) and the top-ranked designs are selected for the next build.

Edit the **Parameters** cell, then Run All. Output: `results/<name>_synthetic_designs.csv`.

**Figures:** the synthetic design library underlies Fig 4 (SrUGT76G1 LD-BT 3) and the UGTSL2 LD-BT 1 analog.

In [ ]:
import os, sys
ROOT = os.path.abspath(os.getcwd())
SRC = os.path.join(ROOT, 'src')
if SRC not in sys.path:
    sys.path.insert(0, SRC)
print('repo root :', ROOT)
print('src on path:', SRC in sys.path)

In [ ]:
import pandas as pd
from synthetic_design import build_residue_sets, generate_constrained_mutations
print('imported synthetic_design')

## Parameters
The published SrUGT76G1 A1-helix LD-BT 3 design run.

In [ ]:
NAME              = 'SrUGT76G1'
LIBRARY           = os.path.join('data', 'SrUGT76G1_variant_library.xlsx')
MOTIF_COL         = 'Motif_Sequence'        # 22-mer A1 candidate motif
ACTIVITY_COL      = 'RSA_RebA_xWT'          # RebA relative specific activity (xWT)
ROUND_COL         = 'Round'
BASE_ROUNDS       = ['LD-BT 0', 'LD-BT 1', 'LD-BT 2', 'Wild type']  # templates
ACTIVITY_THRESHOLD = 1.0      # residue sets from variants strictly above this (xWT)
N_VARIANTS        = 1000      # designs to sample
SEED              = 42
MAX_MUTATIONS     = 7         # upper bound on substitutions per design
OUTPUT            = os.path.join('results', NAME + '_synthetic_designs.csv')
print('library   :', LIBRARY)
print('n_variants:', N_VARIANTS, '| seed:', SEED, '| max_mutations:', MAX_MUTATIONS)

## Load library and apply the Cycle 1.4 filter
LD-BT 3 is restricted to predicted-active variants (`Class_Predictions == 1`); other rounds are unaffected.

In [ ]:
df = pd.read_excel(LIBRARY)
r3_fail = (df[ROUND_COL] == 'LD-BT 3') & (df['Class_Predictions'] != 1)
df = df[~r3_fail].copy()
df = df.dropna(subset=[MOTIF_COL, ACTIVITY_COL])
df[ACTIVITY_COL] = df[ACTIVITY_COL].astype(float)
print('rows after filter:', len(df))
print(df[ROUND_COL].value_counts())

## Templates and residue sets
Templates = motifs from the prior rounds plus WT. Residue sets are built from every variant at or above WT activity.

In [ ]:
base_sequences = df[df[ROUND_COL].isin(BASE_ROUNDS)][MOTIF_COL].tolist()
print('base templates:', len(base_sequences), '(unique %d)' % len(set(base_sequences)))

active = df[df[ACTIVITY_COL] >= ACTIVITY_THRESHOLD]
e_sets = build_residue_sets(active[MOTIF_COL].tolist(), active[ACTIVITY_COL].tolist(),
                            threshold=ACTIVITY_THRESHOLD)
print('positions in residue set:', len(e_sets))
print('variable positions (>1 residue):',
      sorted(p for p, r in e_sets.items() if len(r) > 1))

## Sample the design library

In [ ]:
wt_motif = df[df['Type'] == 'Wild Type'][MOTIF_COL].iloc[0]
designs = generate_constrained_mutations(
    wt_seq=wt_motif, residue_sets=e_sets, n_variants=N_VARIANTS,
    seed=SEED, max_mutations=MAX_MUTATIONS, base_sequences=base_sequences)
print('designs sampled :', len(designs))
print('unique motifs   :', designs['Candidate_Sequence'].nunique())
print('mutation counts :')
print(designs['num_mutations'].value_counts().sort_index())

## Rebuild full sequences and write
The constant flanks outside the 22-mer motif are read from the wild-type row, then the sampled motif is spliced back in.

In [ ]:
wt_full = df[df['Type'] == 'Wild Type']['Full_Sequence'].iloc[0]
i = wt_full.index(wt_motif)
nterm, cterm = wt_full[:i], wt_full[i + len(wt_motif):]
designs['Full_Sequence'] = nterm + designs['Candidate_Sequence'] + cterm
designs.insert(0, 'Design_ID', ['%s-A1-SYN-%04d' % (NAME, k + 1) for k in range(len(designs))])

os.makedirs(os.path.dirname(OUTPUT), exist_ok=True)
designs.to_csv(OUTPUT, index=False)
print('written to :', OUTPUT, '(%d rows)' % len(designs))
print('note: the published table dedups identical motifs (1,000 -> 944 unique) and scores\n'
      'them with the two-stage model; see data/supplementary/A1_LDBT3_synthetic_designs.xlsx')